In [ ]:
import os
import zipfile
import warnings

import numpy as np
import pandas as pd

import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    confusion_matrix,
    classification_report,
    accuracy_score
)

import tensorflow as tf

from tensorflow.keras import layers
from tensorflow.keras.models import Sequential
from tensorflow.keras.callbacks import EarlyStopping
from tensorflow.keras.applications import EfficientNetB0

warnings.filterwarnings("ignore")

np.random.seed(42)
tf.random.set_seed(42)

print(tf.__version__)

In [ ]:
os.makedirs("project_outputs", exist_ok=True)
os.makedirs("project_outputs/tables", exist_ok=True)
os.makedirs("project_outputs/figures", exist_ok=True)
os.makedirs("project_outputs/models", exist_ok=True)

print("Folders created")

In [ ]:
DATASET_PATH = "/kaggle/input/competitions/paddy-disease-classification"

train_csv = os.path.join(DATASET_PATH, "train.csv")
images_dir = os.path.join(DATASET_PATH, "train_images")

df = pd.read_csv(train_csv)

print(df.shape)
df.head()

In [ ]:
class_dist = df["label"].value_counts()

class_dist.to_csv(
    "project_outputs/tables/class_distribution.csv"
)

class_dist

In [ ]:
import matplotlib.pyplot as plt

class_dist = df["label"].value_counts()

plt.figure(figsize=(12,6))

class_dist.plot(kind="bar")

plt.title("Class Distribution")
plt.xlabel("Disease Class")
plt.ylabel("Number of Images")

plt.tight_layout()

plt.show()

In [ ]:
import os

os.makedirs("project_outputs/figures", exist_ok=True)

class_dist = df["label"].value_counts()

plt.figure(figsize=(12,6))

class_dist.plot(kind="bar")

plt.title("Class Distribution")
plt.xlabel("Disease Class")
plt.ylabel("Number of Images")

plt.tight_layout()

plt.savefig(
    "project_outputs/figures/class_distribution.pdf",
    bbox_inches="tight"
)

plt.close()

In [ ]:
train_df, temp_df = train_test_split(
    df,
    test_size=0.20,
    stratify=df["label"],
    random_state=42
)

val_df, test_df = train_test_split(
    temp_df,
    test_size=0.50,
    stratify=temp_df["label"],
    random_state=42
)

print(len(train_df))
print(len(val_df))
print(len(test_df))

In [ ]:
IMG_SIZE = 224
BATCH_SIZE = 32

train_datagen = tf.keras.preprocessing.image.ImageDataGenerator(
    rescale=1./255,
    rotation_range=20,
    width_shift_range=0.2,
    height_shift_range=0.2,
    zoom_range=0.2,
    horizontal_flip=True
)

test_datagen = tf.keras.preprocessing.image.ImageDataGenerator(
    rescale=1./255
)

In [ ]:
# Create correct relative image paths
df["filepath"] = df["label"] + "/" + df["image_id"]

# Recreate train/validation/test splits
train_df, temp_df = train_test_split(
    df,
    test_size=0.20,
    stratify=df["label"],
    random_state=42
)

val_df, test_df = train_test_split(
    temp_df,
    test_size=0.50,
    stratify=temp_df["label"],
    random_state=42
)

print("Train:", len(train_df))
print("Validation:", len(val_df))
print("Test:", len(test_df))

# Data generators
train_gen = train_datagen.flow_from_dataframe(
    dataframe=train_df,
    directory=images_dir,
    x_col="filepath",
    y_col="label",
    target_size=(IMG_SIZE, IMG_SIZE),
    batch_size=BATCH_SIZE,
    class_mode="categorical",
    shuffle=True
)

val_gen = test_datagen.flow_from_dataframe(
    dataframe=val_df,
    directory=images_dir,
    x_col="filepath",
    y_col="label",
    target_size=(IMG_SIZE, IMG_SIZE),
    batch_size=BATCH_SIZE,
    class_mode="categorical",
    shuffle=False
)

test_gen = test_datagen.flow_from_dataframe(
    dataframe=test_df,
    directory=images_dir,
    x_col="filepath",
    y_col="label",
    target_size=(IMG_SIZE, IMG_SIZE),
    batch_size=BATCH_SIZE,
    class_mode="categorical",
    shuffle=False
)

print("\nClass Mapping:")
print(train_gen.class_indices)

print("\nTraining Samples:", train_gen.samples)
print("Validation Samples:", val_gen.samples)
print("Test Samples:", test_gen.samples)

In [ ]:
NUM_CLASSES = len(train_gen.class_indices)

cnn_model = Sequential([

    layers.Input(shape=(224, 224, 3)),

    layers.Conv2D(32, (3,3), activation='relu'),
    layers.MaxPooling2D(2,2),

    layers.Conv2D(64, (3,3), activation='relu'),
    layers.MaxPooling2D(2,2),

    layers.Conv2D(128, (3,3), activation='relu'),
    layers.MaxPooling2D(2,2),

    layers.Conv2D(256, (3,3), activation='relu'),
    layers.MaxPooling2D(2,2),

    layers.Flatten(),

    layers.Dense(256, activation='relu'),

    layers.Dropout(0.5),

    layers.Dense(
        NUM_CLASSES,
        activation='softmax'
    )

])

cnn_model.compile(
    optimizer='adam',
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

cnn_model.summary()

In [ ]:
early_stop = EarlyStopping(
    monitor='val_loss',
    patience=5,
    restore_best_weights=True
)

In [ ]:
cnn_history = cnn_model.fit(
    train_gen,
    validation_data=val_gen,
    epochs=20,
    callbacks=[early_stop]
)

In [ ]:
cnn_model.save(
    "project_outputs/models/cnn_model.keras"
)

In [ ]:
plt.figure(figsize=(12,5))

plt.plot(
    cnn_history.history['accuracy'],
    label='Train Accuracy'
)

plt.plot(
    cnn_history.history['val_accuracy'],
    label='Validation Accuracy'
)

plt.title("CNN Accuracy")
plt.legend()

plt.savefig(
    "project_outputs/figures/cnn_accuracy_curve.pdf",
    bbox_inches='tight'
)

plt.show()

In [ ]:
plt.figure(figsize=(12,5))

plt.plot(
    cnn_history.history['loss'],
    label='Train Loss'
)

plt.plot(
    cnn_history.history['val_loss'],
    label='Validation Loss'
)

plt.title("CNN Loss")
plt.legend()

plt.savefig(
    "project_outputs/figures/cnn_loss_curve.pdf",
    bbox_inches='tight'
)

plt.show()

In [ ]:
cnn_probs = cnn_model.predict(
    test_gen
)

cnn_preds = np.argmax(
    cnn_probs,
    axis=1
)

true_labels = test_gen.classes

In [ ]:
from sklearn.metrics import accuracy_score

cnn_acc = accuracy_score(
    true_labels,
    cnn_preds
)

print("CNN Accuracy:", cnn_acc)

In [ ]:
cnn_report = classification_report(
    true_labels,
    cnn_preds,
    target_names=list(
        test_gen.class_indices.keys()
    ),
    output_dict=True
)

cnn_report_df = pd.DataFrame(
    cnn_report
).transpose()

cnn_report_df.to_csv(
    "project_outputs/tables/cnn_classification_report.csv"
)

cnn_report_df

In [ ]:
cm = confusion_matrix(
    true_labels,
    cnn_preds
)

plt.figure(figsize=(12,10))

sns.heatmap(
    cm,
    annot=True,
    fmt='d',
    cmap='Blues'
)

plt.title("CNN Confusion Matrix")

plt.savefig(
    "project_outputs/figures/cnn_confusion_matrix.pdf",
    bbox_inches='tight'
)

plt.show()

In [ ]:
from tensorflow.keras.applications import EfficientNetB0
from tensorflow.keras.models import Model
from tensorflow.keras.layers import Dense
from tensorflow.keras.layers import GlobalAveragePooling2D
from tensorflow.keras.layers import Dropout

In [ ]:
base_model = EfficientNetB0(
    weights="imagenet",
    include_top=False,
    input_shape=(224,224,3)
)

base_model.trainable = False

x = base_model.output

x = GlobalAveragePooling2D()(x)

x = Dense(
    256,
    activation="relu"
)(x)

x = Dropout(0.5)(x)

output = Dense(
    len(train_gen.class_indices),
    activation="softmax"
)(x)

efficientnet_model = Model(
    inputs=base_model.input,
    outputs=output
)

efficientnet_model.compile(
    optimizer="adam",
    loss="categorical_crossentropy",
    metrics=["accuracy"]
)

efficientnet_model.summary()

In [ ]:
efficientnet_history = efficientnet_model.fit(
    train_gen,
    validation_data=val_gen,
    epochs=10,
    callbacks=[early_stop]
)

In [ ]:
efficientnet_model.save(
    "project_outputs/models/efficientnet_model.keras"
)

In [ ]:
plt.figure(figsize=(12,5))

plt.plot(
    efficientnet_history.history["accuracy"],
    label="Train Accuracy"
)

plt.plot(
    efficientnet_history.history["val_accuracy"],
    label="Validation Accuracy"
)

plt.title("EfficientNetB0 Accuracy")

plt.legend()

plt.savefig(
    "project_outputs/figures/efficientnet_accuracy_curve.pdf",
    bbox_inches="tight"
)

plt.show()

In [ ]:
plt.figure(figsize=(12,5))

plt.plot(
    efficientnet_history.history["loss"],
    label="Train Loss"
)

plt.plot(
    efficientnet_history.history["val_loss"],
    label="Validation Loss"
)

plt.title("EfficientNetB0 Loss")

plt.legend()

plt.savefig(
    "project_outputs/figures/efficientnet_loss_curve.pdf",
    bbox_inches="tight"
)

plt.show()

In [ ]:
efficientnet_probs = efficientnet_model.predict(
    test_gen
)

efficientnet_preds = np.argmax(
    efficientnet_probs,
    axis=1
)

In [ ]:
efficientnet_acc = accuracy_score(
    true_labels,
    efficientnet_preds
)

print(
    "EfficientNet Accuracy:",
    efficientnet_acc
)

In [ ]:
efficientnet_report = classification_report(
    true_labels,
    efficientnet_preds,
    target_names=list(
        test_gen.class_indices.keys()
    ),
    output_dict=True
)

efficientnet_report_df = pd.DataFrame(
    efficientnet_report
).transpose()

efficientnet_report_df.to_csv(
    "project_outputs/tables/efficientnet_classification_report.csv"
)

efficientnet_report_df

In [ ]:
cm_eff = confusion_matrix(
    true_labels,
    efficientnet_preds
)

plt.figure(figsize=(12,10))

sns.heatmap(
    cm_eff,
    annot=True,
    fmt="d",
    cmap="Greens"
)

plt.title(
    "EfficientNetB0 Confusion Matrix"
)

plt.savefig(
    "project_outputs/figures/efficientnet_confusion_matrix.pdf",
    bbox_inches="tight"
)

plt.show()

In [ ]:
comparison_df = pd.DataFrame({
    "Model":[
        "Custom CNN",
        "EfficientNetB0"
    ],
    "Accuracy":[
        cnn_acc,
        efficientnet_acc
    ]
})

comparison_df

In [ ]:
comparison_df.to_csv(
    "project_outputs/tables/model_comparison.csv",
    index=False
)

In [ ]:
comparison_df.to_csv(
    "project_outputs/tables/model_comparison.csv",
    index=False
)

In [ ]:
import shutil

shutil.make_archive(
    "Rice_Leaf_Disease_Project_Outputs",
    "zip",
    "project_outputs"
)

print(
    "ZIP file created successfully."
)

In [ ]:
import os
print(os.getcwd())